# FKG-KT: GPU runs for the Neurocomputing revision

Runs the three jobs that need a GPU: **AKT** (full regime, 8 corpora × 3 seeds), **GKT** in the full regime (8 corpora × 3 seeds) and **GKT** in the sparse regime on the two corpora with more than 150 KCs (ASSISTments 2012, Junyi). Data, splits and graphs are shipped in `kt_colab.zip`, identical to the CPU runs, so the results merge directly.

1. Runtime → Change runtime type → **T4 GPU** (or better).
2. Put `kt_colab.zip` in Google Drive at `MyDrive/kt_colab.zip` (or upload it in cell 3).
3. Run all cells. Results are written to `MyDrive/kt_gpu_runs/` after every run, so a disconnected session resumes where it stopped: just run all cells again.
4. At the end, download `kt_gpu_results.zip` and send it back.

In [ ]:
!nvidia-smi

In [ ]:
!pip -q install python-igraph==1.0.0 leidenalg==0.12.0 numba pyarrow scikit-learn

In [ ]:
import os, zipfile
from google.colab import drive
drive.mount('/content/drive')
ZIP='/content/drive/MyDrive/kt_colab.zip'
if not os.path.exists(ZIP):
    from google.colab import files
    up=files.upload(); ZIP=list(up)[0]
if not os.path.exists('/content/work/kt/src/train.py'):
    zipfile.ZipFile(ZIP).extractall('/content')
RUNS='/content/drive/MyDrive/kt_gpu_runs'
os.makedirs(RUNS,exist_ok=True)
os.environ['RUNS']=RUNS
print(os.listdir('/content/work/kt'))

In [ ]:
# smoke test: one epoch of AKT and GKT on ASSISTments 2009 (written to a scratch folder, not to the results)
import subprocess, os
env={**os.environ,'RUNS':'/content/smoke','NT':'2'}
for m in ['akt','gkt']:
    r=subprocess.run(['python3','/content/work/kt/src/train.py','--data','assist09','--frac','0.1','--model',m,'--epochs','1','--seed','99'],capture_output=True,text=True,env=env)
    print(r.stdout[-300:], r.stderr[-1500:])

In [ ]:
# main run: resumable; each finished run is a JSON + prediction file in MyDrive/kt_gpu_runs
import subprocess, os, time
jobs=[l.split() for l in open('/content/work/kt/jobs_gpu.txt') if l.strip()]
for i,j in enumerate(jobs):
    t=time.time()
    r=subprocess.run(['python3','/content/work/kt/src/train.py']+j,capture_output=True,text=True,env={**os.environ,'NT':'2'})
    msg=(r.stdout.strip() or 'ERR '+' '.join(j)+' '+r.stderr[-800:])
    open(os.environ['RUNS']+'/runner.log','a').write(msg+'\n')
    print(f'[{i+1}/{len(jobs)}] {time.time()-t:.0f}s', msg[:160], flush=True)

In [ ]:
# package results
import shutil
shutil.make_archive('/content/kt_gpu_results','zip',os.environ['RUNS'])
from google.colab import files
files.download('/content/kt_gpu_results.zip')